# Doublons dans les logs de visite bokeh — IP 195.154.187.0

Ce notebook reprend, étape par étape, l'investigation qui a permis d'identifier et de quantifier un biais de tracking Matomo sur le portail bokeh (Médiathèque de Roubaix) : certaines visites sont trackées **côté serveur** plutôt que par le navigateur de l'usager, ce qui peut faire compter deux fois une même visite réelle.

**Déroulé :**
1. Chargement et premier coup d'œil sur les logs
2. Détection de l'anomalie : une IP concentre l'essentiel du trafic
3. Caractérisation du pattern → pas un bot, un tracking côté serveur
4. Généralisation de la détection (`is_server_tracked_visit`)
5. Preuve concrète du double comptage sur un cas réel
6. Recoupement par contenu à l'échelle du fichier (méthode 1)
7. Bootstrap par IP réelle déjà confirmée (méthode 2)
8. Test de significativité (faux positifs par tirage aléatoire)
9. Plancher ajusté
10. Plafond extrapolé
11. Synthèse

**Données :** `data/bokeh_visites_detail.csv` (à la racine du dépôt) (logs bruts Matomo `Live.getLastVisitsDetails`, une ligne par visite — cf. [`data_matomo_logs_bokeh.py`](data_matomo_logs_bokeh.py)).

*Notebook prévu pour être relancé sur un CSV à jour : tous les chiffres se recalculent à l'exécution, rien n'est figé en dur.*

In [1]:
import csv
import json
import os
import random
import re
import statistics
import sys
from collections import Counter, defaultdict

csv.field_size_limit(sys.maxsize)  # actionDetails peut depasser la limite par defaut du module csv

CSV_PATH = os.path.join("..", "data", "bokeh_visites_detail.csv")
print(f"Fichier : {CSV_PATH}")
print(f"Taille  : {os.path.getsize(CSV_PATH) / 1e6:.0f} Mo")

Fichier : data/bokeh_visites_detail.csv
Taille  : 565 Mo


## 1. Premier coup d'œil

Une ligne = une visite (session Matomo). On regarde la volumétrie et la période couverte.

In [2]:
with open(CSV_PATH, newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    n_rows = 0
    dates = []
    for row in reader:
        n_rows += 1
        dates.append(row["serverDate"])

print(f"{n_rows} visites")
print(f"Periode : {min(dates)} -> {max(dates)}")
print(f"{len(fieldnames)} colonnes")

136321 visites
Periode : 2026-03-30 -> 2026-09-16
63 colonnes


## 2. Détection de l'anomalie

`visitIp` est anonymisée par Matomo (dernier octet toujours à `.0` → chaque valeur représente une plage /24). On regarde la répartition du trafic par plage.

In [3]:
ip_counts = Counter()
with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        ip_counts[row["visitIp"]] += 1

total = sum(ip_counts.values())
print(f"{len(ip_counts)} plages /24 distinctes\n")
print(f"{'plage /24':20s} {'visites':>8s} {'%':>6s}")
for ip, c in ip_counts.most_common(10):
    print(f"{ip:20s} {c:8d} {100*c/total:5.1f}%")

7178 plages /24 distinctes

plage /24             visites      %
195.154.187.0           52680  38.6%
62.102.244.0             6501   4.8%
135.125.109.0            2959   2.2%
77.130.243.0             2337   1.7%
77.131.12.0              2003   1.5%
92.184.123.0             1458   1.1%
37.66.149.0              1175   0.9%
92.184.121.0             1167   0.9%
92.184.120.0             1062   0.8%
37.66.148.0              1016   0.7%


La première plage concentre une part anormale du trafic (~38-40%). On la caractérise avant de conclure quoi que ce soit — pourrait être un vrai pic d'usage, un bot, ou autre chose.

## 3. Caractérisation du pattern

On regarde `resolution`, `plugins`, `referrerType` et les types de hits dans `actionDetails` pour la plage la plus active.

In [4]:
TOP_IP = ip_counts.most_common(1)[0][0]

resolutions = Counter()
plugins_empty = 0
referrer_types = Counter()
action_types = Counter()
browsers = Counter()
userid_present = 0
n = 0

with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        if row["visitIp"] != TOP_IP:
            continue
        n += 1
        resolutions[row["resolution"]] += 1
        if not row["plugins"]:
            plugins_empty += 1
        referrer_types[row["referrerType"]] += 1
        browsers[row["browserName"]] += 1
        if row["userId"]:
            userid_present += 1
        for a in json.loads(row["actionDetails"] or "[]"):
            action_types[a.get("type")] += 1

print(f"{n} visites sur {TOP_IP}\n")
print("resolution        :", resolutions.most_common(3))
print(f"plugins vides     : {plugins_empty}/{n} ({100*plugins_empty/n:.0f}%)")
print("referrerType      :", referrer_types.most_common())
print("types d'action    :", action_types.most_common())
print("navigateurs (top8):", browsers.most_common(8))
print(f"userId renseigne  : {userid_present}/{n} ({100*userid_present/n:.0f}%)")

52680 visites sur 195.154.187.0

resolution        : [('unknown', 52680)]
plugins vides     : 52680/52680 (100%)
referrerType      : [('direct', 52637), ('search', 26), ('website', 15), ('social', 2)]
types d'action    : [('event', 131926)]
navigateurs (top8): [('Chrome', 13310), ('Chrome Mobile', 12104), ('Firefox', 11334), ('Microsoft Edge', 3801), ('Mobile Safari', 3529), ('Safari', 3148), ('Samsung Browser', 1540), ('Chrome Mobile iOS', 1408)]
userId renseigne  : 45349/52680 (86%)


→ Ce n'est **pas un bot** (un script aurait un profil homogène, pas cette diversité). L'explication : le portail déclenche un appel de tracking Matomo **côté serveur** (pas depuis le navigateur) lors d'une action nécessitant d'être authentifié (typiquement réserver un document).

**Précision (vérifiée le 2026-09-17)** : cette instance Matomo tourne **sans cookie** — un appel direct à `matomo.php` (`curl`, hors navigateur) ne renvoie aucun `Set-Cookie`, c'est une configuration serveur volontaire, pas un blocage navigateur. L'identification des visiteurs repose donc, pour tout le site, sur le fallback `config_id` (empreinte IP + navigateur + plugins). L'appel serveur n'ayant ni la même IP (le serveur applicatif) ni la même empreinte (pas de JS exécuté → plugins vides) que le vrai navigateur, son `config_id` diverge nécessairement → Matomo lui attribue un `visitorId`/`idVisit` **distincts** de la visite de navigation en cours.


## 4. Généraliser la détection

Plutôt qu'une IP ou un mot-clé de provider en dur (fragile), on retient une signature **comportementale** — elle doit généraliser à d'éventuelles autres IP de sortie du même serveur.

In [5]:
def is_server_tracked_visit(row):
    """Heuristique comportementale, independante de l'IP/provider : la visite
    a ete trackee cote serveur plutot que par le navigateur de l'usager."""
    if row.get("resolution") != "unknown" or row.get("plugins"):
        return False
    if row.get("referrerType") != "direct":
        return False
    ad_raw = row.get("actionDetails") or "[]"
    try:
        ad = json.loads(ad_raw) if ad_raw else []
    except json.JSONDecodeError:
        return False
    if not ad:
        return False
    return all(a.get("type") == "event" for a in ad)

flagged_ip_counts = Counter()
n_flagged = 0
n_total = 0
with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        n_total += 1
        if is_server_tracked_visit(row):
            n_flagged += 1
            flagged_ip_counts[row["visitIp"]] += 1

print(f"{n_flagged}/{n_total} visites flaggees ({100*n_flagged/n_total:.1f}%)")
print("Plages IP concernees :", flagged_ip_counts.most_common())

55596/136321 visites flaggees (40.8%)
Plages IP concernees : [('195.154.187.0', 52637), ('135.125.109.0', 2959)]


La signature comportementale capture bien **plusieurs plages IP** (pas seulement la plus fréquente) — cohérent avec le même serveur d'authentification dont l'IP de sortie a changé sur certaines périodes. Cette fonction est celle utilisée en production dans [`kiblib/utils/matomo.py`](kiblib/utils/matomo.py).

## 5. Preuve concrète du double comptage

On prend une visite serveur qui référence une notice précise (URL `/id/<n>` ou `/record_id/<n>`), et on cherche une visite de navigation normale (autre IP) qui consulte la **même notice** à quelques minutes d'écart.

In [6]:
ID_RE = re.compile(r"/(?:id|record_id)/(\d{5,})")

def extract_ids(action_details):
    ids = set()
    for a in action_details:
        url = a.get("url") or ""
        for m in ID_RE.finditer(url):
            ids.add(m.group(1))
    return ids

example = None
with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        if not is_server_tracked_visit(row) or not row.get("userId"):
            continue
        ids = extract_ids(json.loads(row["actionDetails"] or "[]"))
        if ids:
            example = {"row": row, "ids": ids}
            break

print("Visite serveur exemple :")
print(f"  userId={example['row']['userId']}  date={example['row']['serverDate']}")
print(f"  fenetre=[{example['row']['firstActionTimestamp']}, {example['row']['lastActionTimestamp']}]")
print(f"  notice(s) referencee(s) : {example['ids']}")

Visite serveur exemple :
  userId=73  date=2026-03-31
  fenetre=[1774993971, 1774994170]
  notice(s) referencee(s) : {'4395360', '4395392', '4395469', '4387036'}


In [7]:
target_id = next(iter(example["ids"]))
target_first = int(example["row"]["firstActionTimestamp"])
target_last = int(example["row"]["lastActionTimestamp"])

with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        if is_server_tracked_visit(row):
            continue
        ad_raw = row["actionDetails"] or "[]"
        if f"/id/{target_id}" not in ad_raw and f"/record_id/{target_id}" not in ad_raw:
            continue
        fa, la = int(row["firstActionTimestamp"]), int(row["lastActionTimestamp"])
        if fa <= target_last + 300 and target_first <= la + 300:
            print("Visite de navigation concurrente trouvee :")
            print(f"  ip={row['visitIp']}  fenetre=[{fa}, {la}]  visitorId={row['visitorId']}")
            chevauche = fa <= target_last and target_first <= la
            print(f"  chevauchement avec la visite serveur : {'OUI (memes instants)' if chevauche else 'proche'}")
            break

Les deux fenêtres temporelles se chevauchent (ou sont à quelques secondes d'écart) : c'est littéralement la **même personne, au même instant**, comptée deux fois par Matomo.

## 6. Recoupement par contenu, à l'échelle du fichier

On généralise la méthode précédente à toutes les visites ciblées qui référencent une notice.

In [8]:
def overlap_gap(fa, la, tf, tl):
    """0 si les fenetres se chevauchent, sinon l'ecart en secondes."""
    return 0 if (fa <= tl and tf <= la) else min(abs(fa - tl), abs(tf - la))

GAP = 300  # tolerance en secondes

targets = []
id_occurrences = defaultdict(list)   # notice_id -> [(ip, first, last), ...] (trafic NON cible)
ip_date_index = defaultdict(list)    # (ip, date) -> [(first, last), ...]   (trafic NON cible)

with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        try:
            fa = int(row.get("firstActionTimestamp") or 0)
            la = int(row.get("lastActionTimestamp") or 0)
        except ValueError:
            continue
        if is_server_tracked_visit(row):
            ad = json.loads(row.get("actionDetails") or "[]")
            targets.append({
                "userId": row.get("userId") or None, "date": row.get("serverDate"),
                "first": fa, "last": la, "ids": extract_ids(ad), "confirmed_ip": None,
            })
        else:
            ip_date_index[(row.get("visitIp"), row.get("serverDate"))].append((fa, la))
            for m in ID_RE.finditer(row.get("actionDetails") or "[]"):
                id_occurrences[m.group(1)].append((row.get("visitIp"), fa, la))

print(f"{len(targets)} visites ciblees (is_server_tracked_visit)")

55596 visites ciblees (is_server_tracked_visit)


In [9]:
def content_match(ids, tf, tl):
    for _id in ids:
        for ip, fa, la in id_occurrences.get(_id, []):
            if overlap_gap(fa, la, tf, tl) <= GAP:
                return ip
    return None

for tv in targets:
    if tv["ids"]:
        tv["confirmed_ip"] = content_match(tv["ids"], tv["first"], tv["last"])

n_content_real = sum(1 for tv in targets if tv["confirmed_ip"])
print(f"Confirmees par contenu : {n_content_real} ({100*n_content_real/len(targets):.1f}% des visites ciblees)")

Confirmees par contenu : 8034 (14.5% des visites ciblees)


## 7. Bootstrap : réutiliser une IP réelle déjà confirmée

Limite de la méthode précédente : elle ne couvre que les visites référençant une notice précise (~40% du cluster). Pour les autres (connexions génériques), on réutilise l'IP réelle déjà confirmée pour ce même compte (`userId`), en cherchant une visite très proche dans le temps sur cette IP, le même jour.

In [10]:
anchors_by_user = defaultdict(list)
for tv in targets:
    if tv["confirmed_ip"] and tv["userId"]:
        anchors_by_user[tv["userId"]].append((tv["first"], tv["confirmed_ip"]))
for anchors in anchors_by_user.values():
    anchors.sort()

def bootstrap_match(ip, date, tf, tl):
    return any(overlap_gap(fa, la, tf, tl) <= GAP for fa, la in ip_date_index.get((ip, date), []))

unconfirmed_with_userid = [
    tv for tv in targets if not tv["confirmed_ip"] and tv["userId"] and anchors_by_user.get(tv["userId"])
]
n_no_anchor = sum(1 for tv in targets if not tv["confirmed_ip"] and tv["userId"] and not anchors_by_user.get(tv["userId"]))
n_no_userid = sum(1 for tv in targets if not tv["confirmed_ip"] and not tv["userId"])

n_bootstrap_real = 0
for tv in unconfirmed_with_userid:
    candidate_ip = min(anchors_by_user[tv["userId"]], key=lambda a: abs(a[0] - tv["first"]))[1]
    tv["bootstrap_ip"] = candidate_ip
    if bootstrap_match(candidate_ip, tv["date"], tv["first"], tv["last"]):
        tv["confirmed_ip"] = candidate_ip
        n_bootstrap_real += 1

n_total_real = n_content_real + n_bootstrap_real
n_total_targets = len(targets)

print(f"Confirmees par bootstrap : {n_bootstrap_real}")
print(f"  sans ancre disponible pour ce compte          : {n_no_anchor}")
print(f"  ancre disponible mais aucune visite proche     : {len(unconfirmed_with_userid) - n_bootstrap_real}")
print(f"  sans userId du tout (compte non identifiable)  : {n_no_userid}")
print()
print(f"TOTAL confirme : {n_total_real}/{n_total_targets} ({100*n_total_real/n_total_targets:.1f}% des visites ciblees)")

Confirmees par bootstrap : 12963
  sans ancre disponible pour ce compte          : 13305
  ancre disponible mais aucune visite proche     : 15255
  sans userId du tout (compte non identifiable)  : 6039

TOTAL confirme : 20997/55596 (37.8% des visites ciblees)


## 8. Test de significativité (faux positifs par tirage aléatoire)

Les deux méthodes reposent sur une proximité temporelle — jusqu'à quel point ces correspondances pourraient-elles être des coïncidences ? On casse volontairement la vraie correspondance (ids mélangés entre visites pour la méthode 1, IP d'ancrage tirée chez un autre compte pour la méthode 2) et on compte combien de "confirmations" surgissent quand même par pur hasard.

In [11]:
random.seed(42)
N_ITER = 20

id_bearing = [tv for tv in targets if tv["ids"]]
null_content_counts = []
for _ in range(N_ITER):
    shuffled = [tv["ids"] for tv in id_bearing]
    random.shuffle(shuffled)
    spurious = sum(
        1 for tv, fake_ids in zip(id_bearing, shuffled)
        if fake_ids is not tv["ids"] and content_match(fake_ids, tv["first"], tv["last"])
    )
    null_content_counts.append(spurious)

anchor_pool = sorted({ip for anchors in anchors_by_user.values() for _, ip in anchors})
null_bootstrap_counts = []
for _ in range(N_ITER):
    spurious = 0
    for tv in unconfirmed_with_userid:
        own_ips = {ip for _, ip in anchors_by_user[tv["userId"]]}
        candidates = [ip for ip in anchor_pool if ip not in own_ips]
        if candidates and bootstrap_match(random.choice(candidates), tv["date"], tv["first"], tv["last"]):
            spurious += 1
    null_bootstrap_counts.append(spurious)

def stats(values):
    return statistics.mean(values), (statistics.stdev(values) if len(values) > 1 else 0.0)

mean_a, std_a = stats(null_content_counts)
mean_b, std_b = stats(null_bootstrap_counts)

print(f"Faux positifs simules (contenu)   : moyenne={mean_a:.1f}  ecart-type={std_a:.1f}  ({100*mean_a/len(id_bearing):.2f}%)")
print(f"Faux positifs simules (bootstrap) : moyenne={mean_b:.1f}  ecart-type={std_b:.1f}  ({100*mean_b/len(unconfirmed_with_userid):.2f}%)")

Faux positifs simules (contenu)   : moyenne=57.8  ecart-type=9.5  (0.27%)
Faux positifs simules (bootstrap) : moyenne=117.8  ecart-type=11.9  (0.42%)


Le taux de faux positifs est quasi négligeable (<0,5%) : les correspondances trouvées ne sont pas du bruit statistique, c'est un vrai signal.

## 9. Plancher ajusté

On retire les faux positifs attendus (mesurés ci-dessus) du chiffre brut, avec une fourchette à ±2 écarts-types.

In [12]:
with open(CSV_PATH, newline="") as f:
    total_file_visits = sum(1 for _ in csv.DictReader(f))

adj_total = (n_content_real - mean_a) + (n_bootstrap_real - mean_b)
adj_lo = (n_content_real - (mean_a + 2*std_a)) + (n_bootstrap_real - (mean_b + 2*std_b))
adj_hi = (n_content_real - max(mean_a - 2*std_a, 0)) + (n_bootstrap_real - max(mean_b - 2*std_b, 0))

print(f"Brut                           : {100*n_total_real/total_file_visits:.1f}% du total")
print(f"Ajuste (point central)         : {100*adj_total/total_file_visits:.1f}% du total")
print(f"Fourchette (+/- 2 ecarts-type) : [{100*adj_lo/total_file_visits:.1f}% ; {100*adj_hi/total_file_visits:.1f}%]")

Brut                           : 15.4% du total
Ajuste (point central)         : 15.3% du total
Fourchette (+/- 2 ecarts-type) : [15.2% ; 15.3%]


## 10. Plafond extrapolé

Le plancher ne peut pas vérifier : (a) les comptes avec `userId` mais sans aucune ancre confirmée, (b) les visites sans `userId` du tout. Plutôt que de supposer 100% de doublons pour ces deux populations (hors-sol), on extrapole le taux de confirmation observé sur la population comparable la plus proche :
- **(a)** taux de confirmation des comptes qui *ont* une ancre (bootstrap), appliqué aux comptes qui n'en ont pas — même population (usagers identifiés), seule différe la disponibilité d'une ancre ;
- **(b)** taux de confirmation global parmi tous les comptes identifiés, appliqué aux visites sans `userId` — proxy plus faible (population différente), faute de mieux.

In [13]:
rate_anchor = n_bootstrap_real / len(unconfirmed_with_userid) if unconfirmed_with_userid else 0.0
rate_identified = n_total_real / (n_total_targets - n_no_userid) if (n_total_targets - n_no_userid) else 0.0

extra_no_anchor = n_no_anchor * rate_anchor
extra_no_userid = n_no_userid * rate_identified
ceiling_total = n_total_real + extra_no_anchor + extra_no_userid

print(f"Taux de confirmation (comptes avec ancre)  : {100*rate_anchor:.1f}%  -> +{extra_no_anchor:.0f} (sur {n_no_anchor} visites sans ancre)")
print(f"Taux de confirmation (comptes identifies)  : {100*rate_identified:.1f}%  -> +{extra_no_userid:.0f} (sur {n_no_userid} visites sans userId)")
print()
print(f"Plafond extrapole : {ceiling_total:.0f}/{total_file_visits} = {100*ceiling_total/total_file_visits:.1f}% du total")

Taux de confirmation (comptes avec ancre)  : 45.9%  -> +6112 (sur 13305 visites sans ancre)
Taux de confirmation (comptes identifies)  : 42.4%  -> +2559 (sur 6039 visites sans userId)

Plafond extrapole : 29668/136321 = 21.8% du total


## 11. Synthèse

In [14]:
print("=" * 64)
print("SYNTHESE")
print("=" * 64)
print(f"Visites totales (fichier)      : {total_file_visits}")
print(f"Visites trackees cote serveur  : {n_total_targets} ({100*n_total_targets/total_file_visits:.1f}%)")
print()
print(f"Plancher (ajuste du bruit)     : {100*adj_total/total_file_visits:.1f}%")
print(f"Plafond extrapole              : {100*ceiling_total/total_file_visits:.1f}%")
print()
print(f"==> Entre {100*adj_total/total_file_visits:.0f}% et {100*ceiling_total/total_file_visits:.0f}% des visites du fichier")
print("    sont probablement des doublons d'une visite deja trackee normalement.")

SYNTHESE
Visites totales (fichier)      : 136321
Visites trackees cote serveur  : 55596 (40.8%)

Plancher (ajuste du bruit)     : 15.3%
Plafond extrapole              : 21.8%

==> Entre 15% et 22% des visites du fichier
    sont probablement des doublons d'une visite deja trackee normalement.


## Pour aller plus loin

- **Le vrai correctif est en amont** : puisqu'il n'existe pas de cookie à transmettre (tracking sans cookie), la piste réaliste est d'appeler `setUserId()` côté client dès l'authentification (aujourd'hui seul l'appel serveur le fait) combiné au réglage serveur Matomo `enable_userid_overwrites_visitorid`, qui permet de rattacher deux visites au même `userId` même avec des identifiants différents (cf. [FAQ Matomo](https://fr.matomo.org/faq/general/how-are-requests-with-a-user-id-tracked/)) — nécessite une intervention chez Biblibre, qui héberge l'instance.
- **En attendant**, `is_server_tracked_visit()` et `MatomoClient.get_daily_stats_deduped()` (dans [`kiblib/utils/matomo.py`](../kiblib/utils/matomo.py)) excluent ces doublons des métriques agrégées alimentées par [`data_matomo.py`](data_matomo.py).
- Ce notebook reprend la même logique que [`adm_analyse_estimation_doublons_bokeh.py`](adm_analyse_estimation_doublons_bokeh.py), sous forme narrée et pédagogique — le script est la version "à lancer tel quel", ce notebook la version "à explorer".
- Limites connues : ~24% des visites ciblées n'ont aucune ancre (compte jamais confirmé par contenu) et restent une zone aveugle structurelle ; le pattern n'a pas été vérifié sur les logs BN-R.